Cell 1 — Load & extract qwen.zip

In [ ]:
import os

In [ ]:
# ── Cell 1: Locate existing kan_qwen files ────────────────────────────────────
from pathlib import Path
import os

KAN_QWEN_DIR = Path(
    "/kaggle/input/datasets/deboparna9das/kan-qwen/qwen/kan_qwen"
)

if not KAN_QWEN_DIR.exists():
    raise FileNotFoundError(
        f"kan_qwen directory not found at:\n{KAN_QWEN_DIR}"
    )

print(f"✅ Found kan_qwen at: {KAN_QWEN_DIR}")

# Show contents
print("\nContents:")
for p in sorted(KAN_QWEN_DIR.iterdir()):
    print(" -", p.name)

Cell 2 — Sanity check the extracted contents

In [ ]:
# ── Cell 2: Verify project structure & flag reusable assets ──────────────────

KAN_QWEN = KAN_QWEN_DIR

expected = [
    "final_results.csv",
    "augmented_dataset_final.json",
    "raw_dataset.json",
    "clip_features.npz",
    "text_embeddings.npy",
    "aug_text_embeddings.npy",
    "KAN_Route_best.pt",
    "MLP_Baseline_best.pt",
    "KAN_Route_Aug_aug_best.pt",
    "MLP_Baseline_Aug_aug_best.pt",
    "training_histories.json",
]

print("Reusable asset check:")

for fname in expected:
    fpath = KAN_QWEN / fname

    if fpath.exists():
        status = "✅ found"
        size_mb = f"{fpath.stat().st_size / 1e6:.1f} MB"
    else:
        status = "❌ MISSING"
        size_mb = "-"

    print(f"  {status:12} {fname:35} {size_mb}")


print("\nFull directory tree:")

for p in sorted(KAN_QWEN.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(KAN_QWEN)}")

Cell 3 — Set up the output directory for this revision's deliverables

In [ ]:
# ── Cell 3: Output scaffold for the new-version deliverables ─────────────────
# Matches the file list from the revision prompt (Parts 2–11), so every
# notebook cell downstream just needs to know which subfolder to save into.
OUT = Path("/kaggle/working/outputs")
for sub in [
    "diagnostics",
    "pixel_clip",
    "human_eval",
    "cross_teacher",
    "latency",
    "interpretability",
    "kan_vs_mlp",
    "manuscript",
]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print("✅ Output scaffold ready at:", OUT)
for p in sorted(OUT.iterdir()):
    print(" ", p.name)

Cell 4 — Carry forward the reusable CSVs from this qwen run

In [ ]:
# ── Cell 4: Copy already-valid results forward into the new output tree ──────
import shutil   # ← add this

REUSED = OUT / "reused_from_previous_run"
REUSED.mkdir(exist_ok=True)
reused_files = {
    KAN_QWEN / "final_results.csv": REUSED / "qwen_final_results.csv",
    KAN_QWEN / "training_histories.json": REUSED / "qwen_training_histories.json",
}
for src, dst in reused_files.items():
    if src.exists():
        shutil.copy(src, dst)
        print(f"✅ carried forward: {dst.name}")
    else:
        print(f"⚠️ skipped (not found): {src.name}")

Final cell — Zip everything for download

In [ ]:
# ── Final cell: Package all outputs + reused CSVs for download ───────────────
import zipfile
from datetime import datetime

def zip_dir(src_dir: Path, zip_path: Path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in src_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(src_dir.parent))
    print(f"✅ {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB)")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# 1. Full bundle: every new-version deliverable produced this run
full_zip = Path(f"/kaggle/working/journal_revision_outputs_{timestamp}.zip")
zip_dir(OUT, full_zip)

# 2. Separate, smaller bundle: just the CSVs/results being carried forward
#    from the previous qwen pipeline — useful to keep distinct so it's clear
#    in the final manuscript which numbers are new vs reused.
reused_zip = Path(f"/kaggle/working/reused_qwen_results_{timestamp}.zip")
zip_dir(REUSED, reused_zip)

print("\nDone. Download from the Kaggle 'Output' tab:")
print(" ", full_zip.name)
print(" ", reused_zip.name)

Cell 5 — Reload the exact assets needed to reproduce results

In [ ]:
import torch

In [ ]:
# ── Cell 5: Load raw data, embeddings, CLIP features (augmented + original) ──
import json, numpy as np

INPUT_DIM = 896
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TOOLS = [
    "object_detection","object_counting","person_detection","face_detection",
    "vehicle_detection","animal_detection","text_detection_ocr","logo_detection",
    "anomaly_detection","semantic_segmentation","instance_segmentation",
    "panoptic_segmentation","background_removal","salient_object_segmentation",
    "matting","text_to_image_generation","image_inpainting","image_outpainting",
    "style_transfer","super_resolution","image_colorization","image_denoising",
    "image_deblurring","depth_estimation","surface_normal_estimation",
    "3d_reconstruction","point_cloud_generation","stereo_matching",
    "image_classification","fine_grained_classification","scene_recognition",
    "facial_expression_recognition","age_gender_estimation","action_recognition",
    "pose_estimation","gaze_estimation","visual_question_answering",
    "image_captioning","visual_reasoning","visual_grounding","relation_detection",
    "attribute_recognition","change_detection","medical_image_segmentation",
    "document_layout_analysis","image_quality_assessment","video_object_tracking",
    "optical_flow_estimation","geo_localization","image_retrieval"
]
NUM_TOOLS = len(TOOLS)  # 50-way output head; only 35 classes have training data
print(f"NUM_TOOLS = {NUM_TOOLS}")

# ── Augmented dataset (39,400 samples — primary "Qwen +aug" setting) ─────────
with open(KAN_QWEN / "augmented_dataset_final.json") as f:
    aug_data = json.load(f)

text_emb_aug = np.load(KAN_QWEN / "aug_text_embeddings.npy")
clip_data    = np.load(KAN_QWEN / "clip_features.npz", allow_pickle=True)
clip_raw     = {int(k): v for k, v in clip_data[clip_data.files[0]].item().items()}

clip_vecs_aug = np.stack([clip_raw[int(r["img_id"])] for r in aug_data])
X_aug = np.concatenate([text_emb_aug, clip_vecs_aug], axis=1).astype(np.float32)
y_aug = np.array([r["tool_id"] for r in aug_data], dtype=np.int64)
print(f"✅ Augmented (pixel-CLIP): X={X_aug.shape}  y={y_aug.shape}")

# ── Original (pre-augmentation) dataset — 9,891 samples ──────────────────────
with open(KAN_QWEN / "raw_dataset.json") as f:
    raw_data = json.load(f)

text_emb_raw = np.load(KAN_QWEN / "text_embeddings_raw.npy")
clip_vecs_raw = np.stack([clip_raw[int(r["img_id"])] for r in raw_data])
X_raw = np.concatenate([text_emb_raw, clip_vecs_raw], axis=1).astype(np.float32)
y_raw = np.array([r["tool_id"] for r in raw_data], dtype=np.int64)
print(f"✅ Original (pixel-CLIP):  X={X_raw.shape}  y={y_raw.shape}")

Cell 6 — Model classes (matching the checkpoints exactly) + dataset/split helper

In [ ]:
# ── Cell 6: Model architectures — must match saved .pt files exactly ─────────
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

if not (Path("/kaggle/working/fast-kan")).exists():
    os.system("git clone https://github.com/ZiyaoLi/fast-kan.git /kaggle/working/fast-kan")
import sys; sys.path.append("/kaggle/working/fast-kan")
from fastkan import FastKAN

class KANRoute(nn.Module):
    """proj: 896→48 (Linear) → FastKAN [48→48→NUM_TOOLS], num_grids=4. ~66,962 params."""
    def __init__(self, input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=50, num_grids=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)
        self.kan  = FastKAN(layers_hidden=[proj_dim] + hidden_dims + [num_tools], num_grids=num_grids)

    def forward(self, x):
        return self.kan(self.norm(self.proj(x.float())))

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class MLPBaseline(nn.Module):
    """proj: 896→96 → [64,32] → NUM_TOOLS. GELU + LayerNorm/BatchNorm + dropout. ~96,434 params."""
    def __init__(self, input_dim=896, proj_dim=96, hidden_dims=[64, 32], num_tools=50, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout)]
        prev = proj_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_tools))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.float())

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class ToolRoutingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


def make_split_loaders(X, y, seed=SEED, batch=256):
    """Recreates the exact 70/15/15 split used originally (same seed + order)."""
    full_ds = ToolRoutingDataset(X, y)
    n = len(full_ds)
    n_train = int(0.70 * n); n_val = int(0.15 * n); n_test = n - n_train - n_val
    train_ds, val_ds, test_ds = random_split(
        full_ds, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(seed)
    )
    return (
        DataLoader(train_ds, batch_size=batch, shuffle=True,  num_workers=2, pin_memory=True),
        DataLoader(val_ds,   batch_size=batch, shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(test_ds,  batch_size=batch, shuffle=False, num_workers=2, pin_memory=True),
        test_ds.indices,   # keep original row indices for the interpretability audit later
    )

print("✅ Model classes + split helper ready")

Cell 7 — Load the already-trained checkpoints (no retraining needed)

In [ ]:
# ── Cell 7: Load trained checkpoints for the augmented (Qwen+aug) setting ────
kan_model = KANRoute(input_dim=INPUT_DIM, proj_dim=48, hidden_dims=[48],
                      num_tools=NUM_TOOLS, num_grids=4).float().to(DEVICE)
mlp_model = MLPBaseline(input_dim=INPUT_DIM, proj_dim=96, hidden_dims=[64, 32],
                         num_tools=NUM_TOOLS).float().to(DEVICE)

kan_model.load_state_dict(torch.load(KAN_QWEN / "KAN_Route_Aug_aug_best.pt", map_location=DEVICE))
mlp_model.load_state_dict(torch.load(KAN_QWEN / "MLP_Baseline_Aug_aug_best.pt", map_location=DEVICE))
kan_model.eval(); mlp_model.eval()

print(f"KAN-Route    params: {kan_model.count_parameters():,}")
print(f"MLP-Baseline params: {mlp_model.count_parameters():,}")

train_loader, val_loader, test_loader, test_indices = make_split_loaders(X_aug, y_aug)
print(f"✅ Test set reconstructed: {len(test_indices)} samples")

Cell 8 — Confusion matrices + per-class report (Part 8 of the prompt)

In [ ]:
# ── Cell 8: Confusion matrices, per-class F1, error analysis ─────────────────
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns

def evaluate_full(model, loader, name):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            logits = model(X_b)
            probs = torch.softmax(logits, dim=-1)
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_labels.extend(y_b.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    top1 = np.mean(np.array(all_preds) == np.array(all_labels))
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    print(f"[{name}] Top-1={top1*100:.2f}%  MacroF1={macro_f1:.4f}")
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

kan_preds, kan_labels, kan_probs = evaluate_full(kan_model, test_loader, "KAN-Route")
mlp_preds, mlp_labels, mlp_probs = evaluate_full(mlp_model, test_loader, "MLP-Baseline")

present_classes = sorted(set(kan_labels.tolist()))
class_names = [TOOLS[i] for i in present_classes]

for name, preds, labels in [("kan", kan_preds, kan_labels), ("mlp", mlp_preds, mlp_labels)]:
    cm = confusion_matrix(labels, preds, labels=present_classes)
    plt.figure(figsize=(16, 14))
    sns.heatmap(cm, xticklabels=class_names, yticklabels=class_names, cmap="Blues", cbar=True)
    plt.title(f"{name.upper()} — Confusion Matrix (Qwen +aug, 35-tool)")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=90); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUT / "pixel_clip" / f"{name}_confusion_matrix.png", dpi=150)
    plt.close()

    report = classification_report(labels, preds, labels=present_classes,
                                    target_names=class_names, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(OUT / "pixel_clip" / f"{name}_per_class_report.csv")

print("✅ Confusion matrices + per-class reports saved to outputs/pixel_clip/")

Cell 9 — Text-only and proxy-visual ablation (completing Part 2's missing arms)

In [ ]:
!pip install -q open_clip_torch

In [ ]:
# ── Cell 9: TextOnly-MiniLM and ProxyVisual-CLIPText variants ────────────────
# The pixel-CLIP variant (Text+PixelVisual) is already trained (Cell 7).
# This cell trains the two comparison arms the prompt asks for, reusing the
# exact same train_model() loop/hyperparameters as the original run.

import open_clip

# Build ProxyVisual-CLIPText embeddings from object-list descriptions
clip_txt_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_txt_model = clip_txt_model.to(DEVICE).eval()
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')

def embed_proxy_text(records, batch_size=256):
    descs = [f"a photo with: {', '.join(r.get('objects', []) or ['unknown'])}" for r in records]
    feats = []
    with torch.no_grad():
        for i in range(0, len(descs), batch_size):
            toks = clip_tokenizer(descs[i:i+batch_size]).to(DEVICE)
            f = clip_txt_model.encode_text(toks)
            f = torch.nn.functional.normalize(f, dim=-1)
            feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)

PROXY_CACHE = OUT / "pixel_clip" / "proxy_clip_text_embeddings_aug.npy"
PROXY_CACHE.parent.mkdir(parents=True, exist_ok=True)

if PROXY_CACHE.exists():
    proxy_vecs_aug = np.load(PROXY_CACHE)
else:
    proxy_vecs_aug = embed_proxy_text(aug_data)
    np.save(PROXY_CACHE, proxy_vecs_aug)
print(f"✅ Proxy-text CLIP embeddings: {proxy_vecs_aug.shape}")

feature_variants = {
    "TextOnly-MiniLM":       (text_emb_aug, 384),
    "ProxyVisual-CLIPText":  (proxy_vecs_aug, 512),
    "Text+ProxyVisual":      (np.concatenate([text_emb_aug, proxy_vecs_aug], axis=1), 896),
    # Text+PixelVisual already trained/evaluated in Cells 7–8
}

def train_model(model, train_loader, val_loader, model_name, epochs=60, lr=1e-3,
                 weight_decay=1e-4, is_kan=False, save_dir=None):
    if is_kan:
        lr, epochs = 5e-4, 80
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs, pct_start=0.1)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_val_acc = 0.0
    best_path = (save_dir or OUT) / f"{model_name}_best.pt"
    for epoch in range(1, epochs + 1):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
                correct += (model(X_b).argmax(-1) == y_b).sum().item(); total += len(y_b)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  [{model_name}] ep{epoch}/{epochs}  val_top1={val_acc:.4f}")
    model.load_state_dict(torch.load(best_path))
    return best_val_acc

ablation_rows = []
for variant_name, (X_variant, dim) in feature_variants.items():
    print(f"\n=== {variant_name} (dim={dim}) ===")
    tr, va, te, _ = make_split_loaders(X_variant, y_aug)
    for arch_name, ModelCls, kw in [
        ("KAN-Route", KANRoute, dict(input_dim=dim, proj_dim=48, hidden_dims=[48], num_tools=NUM_TOOLS, num_grids=4)),
        ("MLP",       MLPBaseline, dict(input_dim=dim, proj_dim=96, hidden_dims=[64,32], num_tools=NUM_TOOLS)),
    ]:
        m = ModelCls(**kw).float().to(DEVICE)
        is_kan = arch_name == "KAN-Route"
        best_val = train_model(m, tr, va, f"{variant_name}_{arch_name}", is_kan=is_kan,
                                save_dir=OUT / "pixel_clip")
        preds, labels, _ = evaluate_full(m, te, f"{variant_name}-{arch_name}")
        macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
        top1 = float(np.mean(preds == labels))
        ablation_rows.append(dict(variant=variant_name, model=arch_name, dim=dim,
                                   params=m.count_parameters(), top1=top1, macro_f1=macro_f1))

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(OUT / "pixel_clip" / "pixel_clip_results.csv", index=False)
print("\n", ablation_df.to_string(index=False))
print("✅ pixel_clip_results.csv saved")

Cell 10 — Fair latency breakdown (Part 5)

In [ ]:
# ── Cell 10: Router-only / encoder+router / cached-image latency ─────────────
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))
for p in sbert.parameters(): p.requires_grad = False

def measure_latency(fn, n_warmup=50, n_measure=200):
    for _ in range(n_warmup): fn()
    torch.cuda.synchronize()
    times = []
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    for _ in range(n_measure):
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    times = np.array(times)
    times = times[(times >= np.percentile(times, 5)) & (times <= np.percentile(times, 95))]
    return times.mean(), times.std()

dummy_x = torch.randn(1, INPUT_DIM).to(DEVICE)
sample_query = "detect all the objects in this photo"
sample_img_id = int(aug_data[0]["img_id"])
cached_img_vec = torch.tensor(clip_raw[sample_img_id], dtype=torch.float32).unsqueeze(0).to(DEVICE)

# 1. Router-only
kan_router_mean, kan_router_std = measure_latency(lambda: kan_model(dummy_x))
mlp_router_mean, mlp_router_std = measure_latency(lambda: mlp_model(dummy_x))

# 2. Cached-image setting: query encoder (MiniLM) + cached CLIP vec + router
def cached_pipeline(model):
    with torch.no_grad():
        t = torch.tensor(sbert.encode([sample_query], normalize_embeddings=True)).to(DEVICE)
        x = torch.cat([t, cached_img_vec], dim=1)
        return model(x)

kan_cached_mean, kan_cached_std = measure_latency(lambda: cached_pipeline(kan_model))
mlp_cached_mean, mlp_cached_std = measure_latency(lambda: cached_pipeline(mlp_model))

latency_rows = [
    dict(model="KAN-Route", stage="router_only",        mean_ms=kan_router_mean, std_ms=kan_router_std),
    dict(model="MLP",       stage="router_only",        mean_ms=mlp_router_mean, std_ms=mlp_router_std),
    dict(model="KAN-Route", stage="cached_image_setting",mean_ms=kan_cached_mean, std_ms=kan_cached_std),
    dict(model="MLP",       stage="cached_image_setting",mean_ms=mlp_cached_mean, std_ms=mlp_cached_std),
]
latency_df = pd.DataFrame(latency_rows)
latency_df.to_csv(OUT / "latency" / "fair_latency_results.csv", index=False)
print(latency_df.to_string(index=False))
print("\n⚠️ Note: full 'encoder+router' latency (live CLIP image encode, not cached) "
      "and the LLM end-to-end comparison need the teacher-model loaded — "
      "measured back in the original notebook's Cell 18 (llm_mean/std), "
      "carry those numbers forward rather than reloading the 7B model here.")

Cell 11 — KAN vs MLP reframing table (Part 6)

In [ ]:
# ── Cell 11: Honest KAN-vs-MLP trade-off table ────────────────────────────────
kan_p, kan_l, _ = evaluate_full(kan_model, test_loader, "KAN-Route (pixel, +aug)")
mlp_p, mlp_l, _ = evaluate_full(mlp_model, test_loader, "MLP (pixel, +aug)")

reframe_row = pd.DataFrame([{
    "setting": "Qwen 35-tool +aug (pixel-CLIP)",
    "kan_top1": np.mean(kan_p == kan_l), "mlp_top1": np.mean(mlp_p == mlp_l),
    "kan_macro_f1": f1_score(kan_l, kan_p, average="macro", zero_division=0),
    "mlp_macro_f1": f1_score(mlp_l, mlp_p, average="macro", zero_division=0),
    "kan_params": kan_model.count_parameters(), "mlp_params": mlp_model.count_parameters(),
    "kan_latency_ms": kan_router_mean, "mlp_latency_ms": mlp_router_mean,
    "interpretability": "KAN=spline curves, MLP=none",
}])
reframe_row.to_csv(OUT / "kan_vs_mlp" / "kan_mlp_final_ablation_table.csv", index=False)
print(reframe_row.T)

with open(OUT / "kan_vs_mlp" / "kan_vs_mlp_reframing.md", "w") as f:
    f.write("# KAN vs MLP — Reframed Trade-off (Qwen +aug, pixel-CLIP)\n\n")
    f.write("MLP is the stronger choice when raw speed and marginal accuracy are the only "
            "criteria. KAN-Route is competitive in accuracy while uniquely offering intrinsic "
            "spline-based explanations, making it preferable when auditability is important.\n\n")
    f.write(reframe_row.T.to_markdown())
print("✅ kan_vs_mlp_reframing.md saved")

Cell 12 — Interpretability audit (Part 7): 30 curated examples

In [ ]:
# ── Cell 12: Interpretability audit cases (correct-easy / correct-ambiguous / wrong) ─
kan_model.eval()
test_ds_indices = test_indices  # original row indices in aug_data / X_aug

def get_case(idx):
    x = torch.tensor(X_aug[idx:idx+1], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        logits = kan_model(x)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    top3_idx = probs.argsort()[::-1][:3]
    gt = int(y_aug[idx])
    pred = int(top3_idx[0])
    margin = probs[top3_idx[0]] - probs[top3_idx[1]]
    return dict(
        row_idx=idx, query=aug_data[idx]["query"], gt_tool=TOOLS[gt], pred_tool=TOOLS[pred],
        correct=(gt == pred), confidence=float(probs[pred]), margin=float(margin),
        top3=[TOOLS[i] for i in top3_idx],
    )

cases = [get_case(i) for i in test_ds_indices]
correct_cases = [c for c in cases if c["correct"]]
wrong_cases   = [c for c in cases if not c["correct"]]

correct_easy      = sorted(correct_cases, key=lambda c: -c["margin"])[:10]
correct_ambiguous = sorted(correct_cases, key=lambda c: c["margin"])[:10]
misclassified     = sorted(wrong_cases, key=lambda c: -c["confidence"])[:10]

audit = correct_easy + correct_ambiguous + misclassified
for c, tag in zip(audit, ["correct_easy"]*10 + ["correct_ambiguous"]*10 + ["misclassified"]*10):
    c["case_type"] = tag
    c["notes"] = ""  # fill in manually with the rationale after review

audit_df = pd.DataFrame(audit)
audit_df.to_csv(OUT / "interpretability" / "interpretability_audit_cases.csv", index=False)
print(f"✅ {len(audit_df)} audit cases saved — fill in 'notes' manually per Part 7 of the prompt.")
audit_df[["case_type", "query", "gt_tool", "pred_tool", "confidence"]].head(10)

Cell 13 — Taxonomy-overlap analysis (Part 4 supporting evidence)

In [ ]:
# ── Cell 13: Confusion between semantically overlapping tool pairs ───────────
overlap_pairs = [
    ("background_removal", "image_inpainting"),
    ("image_captioning", "visual_question_answering"),
    ("object_detection", "image_classification"),
    ("semantic_segmentation", "matting"),
    ("scene_recognition", "image_captioning"),
    ("instance_segmentation", "panoptic_segmentation"),
]

cm_full = confusion_matrix(kan_labels, kan_preds, labels=list(range(NUM_TOOLS)))
rows = []
for a, b in overlap_pairs:
    ia, ib = TOOLS.index(a), TOOLS.index(b)
    rows.append(dict(
        pair=f"{a} vs {b}",
        a_to_b_confusions=int(cm_full[ia, ib]),
        b_to_a_confusions=int(cm_full[ib, ia]),
        support_a=int(cm_full[ia].sum()), support_b=int(cm_full[ib].sum()),
    ))
overlap_df = pd.DataFrame(rows)
overlap_df.to_csv(OUT / "diagnostics" / "taxonomy_overlap_analysis.csv", index=False)
print(overlap_df.to_string(index=False))

Updated final cell — zip, including all reused CSVs

In [ ]:
# ── Final cell: Package everything, keeping "reused" vs "new" separable ──────
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

def zip_dir(src_dir: Path, zip_path: Path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in src_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(src_dir.parent))
    print(f"✅ {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB)")

# Reused CSVs from the original qwen run (unchanged, carried forward as evidence)
reused_files = {
    KAN_QWEN / "final_results.csv": REUSED / "qwen_final_results.csv",
    KAN_QWEN / "training_histories.json": REUSED / "qwen_training_histories.json",
}
for src, dst in reused_files.items():
    if src.exists():
        shutil.copy(src, dst)

full_zip   = Path(f"/kaggle/working/qwen_journal_revision_outputs_{timestamp}.zip")
reused_zip = Path(f"/kaggle/working/qwen_reused_results_{timestamp}.zip")

zip_dir(OUT, full_zip)
zip_dir(REUSED, reused_zip)

print("\nStill pending (needs Llama + human-query data, not qwen-only):")
print("  - cross_teacher_results.csv (train Qwen → test Llama/human)")
print("  - human_eval_results.csv")
print("\nDownload from the Kaggle 'Output' tab:")
print(" ", full_zip.name)
print(" ", reused_zip.name)

Next cells — diagnose and fix this before anything else

In [ ]:
# ── Cell 14: Diagnose the pixel-CLIP reload discrepancy ──────────────────────
# Checks whether X_aug (Cell 5) actually matches what the checkpoint expects.

# 1. Coverage check — does every row's img_id actually exist in clip_raw,
#    or are we silently pairing some rows with the wrong / missing vector?
missing_ids = [r["img_id"] for r in aug_data if int(r["img_id"]) not in clip_raw]
print(f"Missing img_ids in clip_raw: {len(missing_ids)} / {len(aug_data)}")

# 2. Distribution check — CLIP image features should be L2-normalized
#    (unit norm, since extract_clip_features() calls F.normalize).
#    If these stats look off, clip_features.npz here isn't what the
#    checkpoint was actually trained on.
norms = np.linalg.norm(clip_vecs_aug, axis=1)
print(f"clip_vecs_aug norm: mean={norms.mean():.4f} std={norms.std():.4f} "
      f"(expect ~1.0 if L2-normalized)")

text_norms = np.linalg.norm(text_emb_aug, axis=1)
print(f"text_emb_aug norm:  mean={text_norms.mean():.4f} std={text_norms.std():.4f}")

# 3. Isolate which half of the fused vector is actually broken —
#    evaluate the loaded checkpoint on TEXT-ONLY-ZEROED and
#    VISUAL-ONLY-ZEROED versions of the same input.
def eval_zeroed(model, X, zero_range, loader_maker=make_split_loaders):
    X_mod = X.copy()
    X_mod[:, zero_range[0]:zero_range[1]] = 0.0
    _, _, te, _ = loader_maker(X_mod, y_aug)
    preds, labels, _ = evaluate_full(model, te, "zeroed-check")
    return float(np.mean(preds == labels))

acc_visual_zeroed = eval_zeroed(kan_model, X_aug, (384, 896))  # keep text, kill visual
acc_text_zeroed   = eval_zeroed(kan_model, X_aug, (0, 384))    # keep visual, kill text

print(f"\nKAN accuracy with visual half zeroed (text-only signal): {acc_visual_zeroed*100:.2f}%")
print(f"KAN accuracy with text half zeroed (visual-only signal):  {acc_text_zeroed*100:.2f}%")
print("\nIf 'visual half zeroed' recovers close to 95%, the visual features")
print("(clip_vecs_aug) are the corrupted/misaligned component.")

In [ ]:
# ── Cell 15: Rebuild text embeddings fresh to rule out a stale/misordered cache ─
# aug_text_embeddings.npy was cached from the ORIGINAL session's df_aug —
# if that dataframe's row order ever differed from augmented_dataset_final.json's
# order, the cached .npy is silently misaligned with aug_data. Recomputing
# directly from aug_data guarantees correct alignment regardless of history.

from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))
for p in sbert.parameters():
    p.requires_grad = False

queries_fresh = [r["query"] for r in aug_data]
text_emb_fresh = sbert.encode(
    queries_fresh, batch_size=512, show_progress_bar=True, normalize_embeddings=True
)

X_aug_fresh = np.concatenate([text_emb_fresh, clip_vecs_aug], axis=1).astype(np.float32)

_, _, test_loader_fresh, _ = make_split_loaders(X_aug_fresh, y_aug)
kan_preds_f, kan_labels_f, _ = evaluate_full(kan_model, test_loader_fresh, "KAN-Route (fresh text)")
mlp_preds_f, mlp_labels_f, _ = evaluate_full(mlp_model, test_loader_fresh, "MLP (fresh text)")

print(f"\nKAN Top-1 with freshly-recomputed text embeddings: "
      f"{np.mean(kan_preds_f == kan_labels_f)*100:.2f}%  (compare to 95.62% target)")

Next cell — test the other checkpoint before considering retraining

In [ ]:
# ── Cell 16: Try the alternate checkpoint (kan_route_aug_final.pt) ───────────
# Hypothesis: "KAN_Route_Aug_aug_best.pt" is a stale artifact from an earlier
# training attempt; "kan_route_aug_final.pt" is the one that actually
# produced the 95.62%/95.69% numbers in final_results.csv.

kan_model_final = KANRoute(input_dim=INPUT_DIM, proj_dim=48, hidden_dims=[48],
                            num_tools=NUM_TOOLS, num_grids=4).float().to(DEVICE)
mlp_model_final = MLPBaseline(input_dim=INPUT_DIM, proj_dim=96, hidden_dims=[64, 32],
                               num_tools=NUM_TOOLS).float().to(DEVICE)

kan_model_final.load_state_dict(torch.load(KAN_QWEN / "kan_route_aug_final.pt", map_location=DEVICE))
mlp_model_final.load_state_dict(torch.load(KAN_QWEN / "mlp_baseline_aug_final.pt", map_location=DEVICE))
kan_model_final.eval(); mlp_model_final.eval()

_, _, test_loader_check, _ = make_split_loaders(X_aug, y_aug)

kan_preds2, kan_labels2, _ = evaluate_full(kan_model_final, test_loader_check, "KAN-Route (final.pt)")
mlp_preds2, mlp_labels2, _ = evaluate_full(mlp_model_final, test_loader_check, "MLP (final.pt)")

kan_top1_2 = np.mean(kan_preds2 == kan_labels2)
mlp_top1_2 = np.mean(mlp_preds2 == mlp_labels2)
print(f"\nKAN Top-1 (kan_route_aug_final.pt): {kan_top1_2*100:.2f}%  (target: 95.62%)")
print(f"MLP Top-1 (mlp_baseline_aug_final.pt): {mlp_top1_2*100:.2f}%  (target: 95.69%)")

if kan_top1_2 > 0.90:
    print("\n✅ Confirmed — use kan_route_aug_final.pt / mlp_baseline_aug_final.pt "
          "as the checkpoints going forward. KAN_Route_Aug_aug_best.pt is stale.")
else:
    print("\n⚠️ Still off target — the split-order theory needs revisiting next "
          "(compare against original notebook's val_acc curve in training_histories.json).")

In [ ]:
import os

In [ ]:
# ── Resume Cell: Rebuild everything from saved zips, no retraining needed ────
from pathlib import Path
import json, shutil, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# 1. Locate your original kan-qwen source data (raw json, embeddings, clip cache)
KAN_QWEN = next(Path("/kaggle/input").rglob("kan_qwen")).resolve() if list(Path("/kaggle/input").rglob("kan_qwen")) else None
if KAN_QWEN is None:
    KAN_QWEN = Path("/kaggle/input/datasets/deboparna9das/kan-qwen/qwen/kan_qwen")
print(f"KAN_QWEN: {KAN_QWEN}  exists={KAN_QWEN.exists()}")

# 2. Locate your PREVIOUS outputs zip (upload it as a new Kaggle Dataset first,
#    named e.g. "qwen-journal-outputs" — then this finds it automatically)
prev_outputs_candidates = list(Path("/kaggle/input").rglob("pixel_clip_results.csv"))
if not prev_outputs_candidates:
    raise FileNotFoundError(
        "Previous outputs not found. Add your qwen_journal_revision_outputs_...zip "
        "as a new Kaggle Dataset (Add Data -> Upload) before running this cell."
    )
PREV_OUT = prev_outputs_candidates[0].parent.parent  # .../outputs/pixel_clip/.. -> .../outputs
print(f"PREV_OUT: {PREV_OUT}")

# 3. Fresh working output scaffold (writable)
OUT = Path("/kaggle/working/outputs")
for sub in ["diagnostics","pixel_clip","human_eval","cross_teacher","latency",
            "interpretability","kan_vs_mlp","manuscript","reused_from_previous_run"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

# 4. Copy forward everything you already computed, so nothing needs recomputing
for sub in ["diagnostics","pixel_clip","interpretability","kan_vs_mlp","latency"]:
    src_sub = PREV_OUT / sub
    if src_sub.exists():
        for f in src_sub.iterdir():
            if f.is_file():
                shutil.copy(f, OUT / sub / f.name)
print("✅ Previous outputs copied into fresh /kaggle/working/outputs")

# 5. Reload raw data + embeddings (cheap, no GPU compute — just file reads)
NUM_TOOLS = 50
TOOLS = [
    "object_detection","object_counting","person_detection","face_detection",
    "vehicle_detection","animal_detection","text_detection_ocr","logo_detection",
    "anomaly_detection","semantic_segmentation","instance_segmentation",
    "panoptic_segmentation","background_removal","salient_object_segmentation",
    "matting","text_to_image_generation","image_inpainting","image_outpainting",
    "style_transfer","super_resolution","image_colorization","image_denoising",
    "image_deblurring","depth_estimation","surface_normal_estimation",
    "3d_reconstruction","point_cloud_generation","stereo_matching",
    "image_classification","fine_grained_classification","scene_recognition",
    "facial_expression_recognition","age_gender_estimation","action_recognition",
    "pose_estimation","gaze_estimation","visual_question_answering",
    "image_captioning","visual_reasoning","visual_grounding","relation_detection",
    "attribute_recognition","change_detection","medical_image_segmentation",
    "document_layout_analysis","image_quality_assessment","video_object_tracking",
    "optical_flow_estimation","geo_localization","image_retrieval"
]

with open(KAN_QWEN / "augmented_dataset_final.json") as f:
    aug_data = json.load(f)

text_emb_aug = np.load(KAN_QWEN / "aug_text_embeddings.npy")
clip_data = np.load(KAN_QWEN / "clip_features.npz", allow_pickle=True)
clip_raw = {int(k): v for k, v in clip_data[clip_data.files[0]].item().items()}
clip_vecs_aug = np.stack([clip_raw[int(r["img_id"])] for r in aug_data])

X_aug = np.concatenate([text_emb_aug, clip_vecs_aug], axis=1).astype(np.float32)
y_aug = np.array([r["tool_id"] for r in aug_data], dtype=np.int64)
print(f"✅ Reloaded X_aug={X_aug.shape}  y_aug={y_aug.shape}")

# 6. Model classes (must match checkpoints exactly — same as before)
if not Path("/kaggle/working/fast-kan").exists():
    os.system("git clone https://github.com/ZiyaoLi/fast-kan.git /kaggle/working/fast-kan")
import sys; sys.path.append("/kaggle/working/fast-kan")
from fastkan import FastKAN

class KANRoute(nn.Module):
    def __init__(self, input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=50, num_grids=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)
        self.kan = FastKAN(layers_hidden=[proj_dim] + hidden_dims + [num_tools], num_grids=num_grids)
    def forward(self, x): return self.kan(self.norm(self.proj(x.float())))
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class MLPBaseline(nn.Module):
    def __init__(self, input_dim=896, proj_dim=96, hidden_dims=[64,32], num_tools=50, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout)]
        prev = proj_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_tools))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x.float())
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class ToolRoutingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def make_split_loaders(X, y, seed=42, batch=256):
    full_ds = ToolRoutingDataset(X, y)
    n = len(full_ds)
    n_train, n_val = int(0.70*n), int(0.15*n)
    n_test = n - n_train - n_val
    train_ds, val_ds, test_ds = random_split(
        full_ds, [n_train, n_val, n_test], generator=torch.Generator().manual_seed(seed))
    return (DataLoader(train_ds, batch_size=batch, shuffle=True),
            DataLoader(val_ds, batch_size=batch, shuffle=False),
            DataLoader(test_ds, batch_size=batch, shuffle=False),
            test_ds.indices)

def evaluate_full(model, loader, name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            preds = model(X_b).argmax(-1)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(y_b.cpu().numpy())
    top1 = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    print(f"[{name}] Top-1={top1*100:.2f}%")
    return np.array(all_preds), np.array(all_labels)

# 7. Reload checkpoints (already-trained — no retraining)
kan_model = KANRoute(input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=NUM_TOOLS, num_grids=4).to(DEVICE)
mlp_model = MLPBaseline(input_dim=896, proj_dim=96, hidden_dims=[64,32], num_tools=NUM_TOOLS).to(DEVICE)
kan_model.load_state_dict(torch.load(KAN_QWEN / "KAN_Route_Aug_aug_best.pt", map_location=DEVICE))
mlp_model.load_state_dict(torch.load(KAN_QWEN / "MLP_Baseline_Aug_aug_best.pt", map_location=DEVICE))
kan_model.eval(); mlp_model.eval()

train_loader, val_loader, test_loader, test_indices = make_split_loaders(X_aug, y_aug)
print("✅ Fully resumed — ready to re-run the Cell 17/18 label-mapping diagnostic.")

In [ ]:
# ── Cell 17: Recover the true tool_id → name mapping from the model itself ───
# The checkpoint is proven accurate (95.75% during its own training). If the
# CURRENT tool_id encoding doesn't match what it was trained on, we can
# recover the correct mapping empirically: for each predicted class index,
# find which TRUE tool name dominates among samples the model assigned to it.
# If the model really is ~95% accurate, this majority vote will be a clean,
# high-confidence signal per index.

import torch
from collections import Counter

_, _, test_loader_diag, test_idx_diag = make_split_loaders(X_aug, y_aug)

kan_model.eval()
all_preds, all_names = [], []
with torch.no_grad():
    for X_b, _ in test_loader_diag:
        X_b = X_b.float().to(DEVICE)
        preds = kan_model(X_b).argmax(-1).cpu().numpy()
        all_preds.extend(preds)

# align predictions back to the original aug_data rows via test_idx_diag
true_names_test = [aug_data[i]["tool"] for i in test_idx_diag]

recovered_map = {}
confidence = {}
for pred_idx in sorted(set(all_preds)):
    names_for_idx = [true_names_test[i] for i, p in enumerate(all_preds) if p == pred_idx]
    if not names_for_idx:
        continue
    counts = Counter(names_for_idx)
    top_name, top_count = counts.most_common(1)[0]
    recovered_map[pred_idx] = top_name
    confidence[pred_idx] = top_count / len(names_for_idx)

print("Recovered predicted_index -> tool_name mapping:")
for idx in sorted(recovered_map):
    print(f"  {idx:>2} -> {recovered_map[idx]:35} (majority confidence: {confidence[idx]*100:.1f}%, n={sum(1 for p in all_preds if p==idx)})")

avg_conf = np.mean(list(confidence.values()))
print(f"\nAverage majority-vote confidence: {avg_conf*100:.1f}%")
print("If this is high (>85%) across most indices, the model is consistent —")
print("confirming the bug is a stale/rebuilt tool_id scheme, not the model itself.")

In [ ]:
# ── Cell 18: Rebuild y_aug from the recovered mapping and re-evaluate ────────
name_to_recovered_id = {name: idx for idx, name in recovered_map.items()}

# sanity check: did every one of the 35 present tool names get recovered?
missing_names = set(r["tool"] for r in aug_data) - set(name_to_recovered_id.keys())
print(f"Tool names not recovered (low-confidence or absent from test set): {missing_names}")

y_aug_fixed = np.array(
    [name_to_recovered_id.get(r["tool"], -1) for r in aug_data], dtype=np.int64
)
valid_mask = y_aug_fixed != -1
print(f"Usable rows after remapping: {valid_mask.sum()} / {len(y_aug_fixed)}")

X_aug_valid = X_aug[valid_mask]
y_aug_valid = y_aug_fixed[valid_mask]

_, _, test_loader_fixed, _ = make_split_loaders(X_aug_valid, y_aug_valid)
kan_preds_fixed, kan_labels_fixed, _ = evaluate_full(kan_model, test_loader_fixed, "KAN-Route (remapped labels)")
mlp_preds_fixed, mlp_labels_fixed, _ = evaluate_full(mlp_model, test_loader_fixed, "MLP (remapped labels)")

print(f"\nKAN Top-1 with recovered label mapping: {np.mean(kan_preds_fixed==kan_labels_fixed)*100:.2f}% (target: 95.62%)")
print(f"MLP Top-1 with recovered label mapping: {np.mean(mlp_preds_fixed==mlp_labels_fixed)*100:.2f}% (target: 95.69%)")

In [ ]:
# ── Cell 19: Rebuild y using the correct label scheme (LabelEncoder on names) ─
from sklearn.preprocessing import LabelEncoder

# Replicate Cell 42's exact filtering + label logic from the original notebook
X_list, y_list, valid_idx = [], [], []
for i, item in enumerate(aug_data):
    img_id = item["img_id"]
    if int(img_id) in clip_raw:
        text_vec = text_emb_aug[i]
        clip_vec = clip_raw[int(img_id)]
        X_list.append(np.concatenate([text_vec, clip_vec]))
        y_list.append(item["tool"])
        valid_idx.append(i)

X_aug_correct = np.array(X_list, dtype=np.float32)

le = LabelEncoder()
y_aug_correct = le.fit_transform(y_list)

print(f"X: {X_aug_correct.shape}  y: {y_aug_correct.shape}")
print(f"Num classes: {len(le.classes_)}")
print(f"Dropped rows: {len(aug_data) - len(X_list)}")
print(f"\nAlphabetical class order (index -> name):")
for idx, name in enumerate(le.classes_):
    print(f"  {idx:>2} -> {name}")

# Re-evaluate the ALREADY-TRAINED checkpoint on the CORRECT label scheme
_, _, test_loader_correct, test_idx_correct = make_split_loaders(X_aug_correct, y_aug_correct)
kan_preds_c, kan_labels_c, _ = evaluate_full(kan_model, test_loader_correct, "KAN-Route (correct labels)")
mlp_preds_c, mlp_labels_c, _ = evaluate_full(mlp_model, test_loader_correct, "MLP (correct labels)")

kan_top1_c = np.mean(kan_preds_c == kan_labels_c)
mlp_top1_c = np.mean(mlp_preds_c == mlp_labels_c)
print(f"\nKAN Top-1: {kan_top1_c*100:.2f}%  (target: 95.62%)")
print(f"MLP Top-1: {mlp_top1_c*100:.2f}%  (target: 95.69%)")

In [ ]:
# ── Cell 20: Decisive test — evaluate on the FULL dataset, no split ──────────
# If the split boundary is the issue, evaluating on everything (train+val+test
# combined) should recover close to ~95%+, since most rows were seen in
# training and leakage can only help. If it's still ~68%, the split isn't
# the cause and something else (checkpoint/features) is still wrong.

full_ds_check = ToolRoutingDataset(X_aug, y_aug)   # tool_id-based labels, full dataset
full_loader_check = DataLoader(full_ds_check, batch_size=256, shuffle=False)

kan_preds_full, kan_labels_full, _ = evaluate_full(kan_model, full_loader_check, "KAN-Route (FULL dataset, tool_id labels)")
mlp_preds_full, mlp_labels_full, _ = evaluate_full(mlp_model, full_loader_check, "MLP (FULL dataset, tool_id labels)")

print(f"\nKAN Top-1 on full dataset: {np.mean(kan_preds_full==kan_labels_full)*100:.2f}%")
print(f"MLP Top-1 on full dataset: {np.mean(mlp_preds_full==mlp_labels_full)*100:.2f}%")
print("\nIf this is close to 95%+, the split boundary/order is the whole problem —")
print("the checkpoint and labels are both fine, we just can't recover the ORIGINAL")
print("exact test split from the exported files alone.")
print("If this is still ~68%, the split isn't the cause — something in the")
print("checkpoint or clip_features.npz itself doesn't match what was trained on.")

In [ ]:
def train_model(model, train_loader, val_loader, model_name, epochs=60, lr=1e-3,
                 weight_decay=1e-4, is_kan=False, save_dir=None):
    if is_kan:
        lr, epochs = 5e-4, 80

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        steps_per_epoch=len(train_loader),
        epochs=epochs,
        pct_start=0.1
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_val_acc = 0.0
    best_path = (save_dir or OUT) / f"{model_name}_best.pt"

    for epoch in range(1, epochs + 1):
        # Training
        model.train()

        for X_b, y_b in train_loader:
            X_b = X_b.float().to(DEVICE)
            y_b = y_b.to(DEVICE)

            optimizer.zero_grad()

            loss = criterion(model(X_b), y_b)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()
            scheduler.step()

        # Validation
        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b = X_b.float().to(DEVICE)
                y_b = y_b.to(DEVICE)

                correct += (
                    model(X_b).argmax(-1) == y_b
                ).sum().item()

                total += len(y_b)

        val_acc = correct / total

        # Save best checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"  [{model_name}] ep{epoch}/{epochs}  "
                f"val_top1={val_acc:.4f}"
            )

    # Restore best model
    model.load_state_dict(torch.load(best_path))

    return best_val_acc

In [ ]:
# ── Cell 21: Retrain KAN-Route and MLP fresh on verified-correct data ────────
# Uses the same architecture, hyperparameters, and X_aug/y_aug (tool_id labels,
# now fully verified) as the original Cell 33 training run.

train_loader_new, val_loader_new, test_loader_new, test_idx_new = make_split_loaders(X_aug, y_aug)

kan_model_v2 = KANRoute(input_dim=896, proj_dim=48, hidden_dims=[48],
                         num_tools=NUM_TOOLS, num_grids=4).float().to(DEVICE)
mlp_model_v2 = MLPBaseline(input_dim=896, proj_dim=96, hidden_dims=[64, 32],
                            num_tools=NUM_TOOLS).float().to(DEVICE)

print("Training KAN-Route (FastKAN, 80 epochs)...")
kan_val_acc = train_model(kan_model_v2, train_loader_new, val_loader_new,
                           "KAN_Route_v2", is_kan=True, save_dir=OUT / "pixel_clip")

print("\nTraining MLP-Baseline (60 epochs)...")
mlp_val_acc = train_model(mlp_model_v2, train_loader_new, val_loader_new,
                           "MLP_Baseline_v2", is_kan=False, epochs=60, save_dir=OUT / "pixel_clip")

print(f"\nBest val_acc — KAN: {kan_val_acc*100:.2f}%  MLP: {mlp_val_acc*100:.2f}%")

# Final held-out test evaluation
kan_preds_v2, kan_labels_v2 = evaluate_full(kan_model_v2, test_loader_new, "KAN-Route v2 (test)")
mlp_preds_v2, mlp_labels_v2 = evaluate_full(mlp_model_v2, test_loader_new, "MLP v2 (test)")

kan_top1_v2 = np.mean(kan_preds_v2 == kan_labels_v2)
mlp_top1_v2 = np.mean(mlp_preds_v2 == mlp_labels_v2)
print(f"\nFinal Top-1 — KAN: {kan_top1_v2*100:.2f}%  MLP: {mlp_top1_v2*100:.2f}%")
print("(Target ballpark: ~95%, matching original final_results.csv)")

In [ ]:
import pandas as pd

In [ ]:
# ── Cell 22: Promote v2 models as the source of truth + regenerate Part 2 table ─
kan_model, mlp_model = kan_model_v2, mlp_model_v2   # simplest fix: rebind the names
                                                      # everything in Cells 8, 11, 12
                                                      # already refers to kan_model/mlp_model

# Save these as the canonical checkpoints going forward
torch.save(kan_model.state_dict(), OUT / "pixel_clip" / "KAN_Route_v2_verified_best.pt")
torch.save(mlp_model.state_dict(), OUT / "pixel_clip" / "MLP_Baseline_v2_verified_best.pt")

# Recompute macro F1 to see if KAN still differentiates on that axis
from sklearn.metrics import f1_score
kan_macro_f1_v2 = f1_score(kan_labels_v2, kan_preds_v2, average="macro", zero_division=0)
mlp_macro_f1_v2 = f1_score(mlp_labels_v2, mlp_preds_v2, average="macro", zero_division=0)
print(f"KAN macro F1: {kan_macro_f1_v2:.4f}")
print(f"MLP macro F1: {mlp_macro_f1_v2:.4f}")

# Update pixel_clip_results.csv with the CORRECTED Text+PixelVisual row,
# completing the full 4-way ablation (Part 2 of the prompt) cleanly
ablation_df = pd.read_csv(OUT / "pixel_clip" / "pixel_clip_results.csv")
pixel_visual_row = pd.DataFrame([
    dict(variant="Text+PixelVisual", model="KAN-Route", dim=896,
         params=kan_model.count_parameters(), top1=float(np.mean(kan_preds_v2==kan_labels_v2)),
         macro_f1=kan_macro_f1_v2),
    dict(variant="Text+PixelVisual", model="MLP", dim=896,
         params=mlp_model.count_parameters(), top1=float(np.mean(mlp_preds_v2==mlp_labels_v2)),
         macro_f1=mlp_macro_f1_v2),
])
ablation_df_full = pd.concat([ablation_df, pixel_visual_row], ignore_index=True)
ablation_df_full.to_csv(OUT / "pixel_clip" / "pixel_clip_results.csv", index=False)

print("\n✅ Full 4-variant ablation table:")
print(ablation_df_full[["variant","model","dim","params","top1","macro_f1"]].to_string(index=False))

In [ ]:
# ── Cell 23: Confirm the macro-F1 gap is a rare-class story, not noise ───────
kan_report = pd.read_csv(OUT / "pixel_clip" / "kan_per_class_report.csv", index_col=0)
mlp_report = pd.read_csv(OUT / "pixel_clip" / "mlp_per_class_report.csv", index_col=0)
# (re-run Cell 8 first with the rebound kan_model/mlp_model so these reflect v2)

merged = kan_report[["f1-score","support"]].join(
    mlp_report[["f1-score"]], lsuffix="_kan", rsuffix="_mlp"
).dropna()
merged["gap"] = merged["f1-score_kan"] - merged["f1-score_mlp"]
merged = merged.sort_values("support")

print("Per-class F1 gap (KAN - MLP), sorted by support (rarest classes first):")
print(merged.head(15).to_string())

low_support_gap = merged[merged["support"] < merged["support"].median()]["gap"].mean()
high_support_gap = merged[merged["support"] >= merged["support"].median()]["gap"].mean()
print(f"\nAvg KAN advantage on below-median-support classes: {low_support_gap:+.3f}")
print(f"Avg KAN advantage on above-median-support classes:  {high_support_gap:+.3f}")

In [ ]:
# ── Cell 24: Updated, stronger reframing writeup ──────────────────────────────
with open(OUT / "kan_vs_mlp" / "kan_vs_mlp_reframing.md", "w") as f:
    f.write("# KAN vs MLP — Reframed Trade-off (Qwen +aug, verified v2 checkpoints)\n\n")
    f.write("Top-1 accuracy is statistically tied between KAN-Route and the MLP baseline "
            "across all tested feature configurations (TextOnly, ProxyVisual, Text+ProxyVisual, "
            "Text+PixelVisual) — differences are within noise of a ~500-sample test set.\n\n")
    f.write("Macro F1 tells a different story: KAN-Route outperforms the MLP baseline by "
            "0.14–0.17 macro-F1 points in every fused/text setting, indicating the MLP achieves "
            "its comparable Top-1 by performing well on frequent classes while underperforming "
            "on rarer ones — exactly the failure mode Top-1 accuracy hides and macro F1 exposes. "
            "KAN-Route's spline-based routing generalizes more evenly across the class distribution.\n\n")
    f.write("Combined with its intrinsic spline-based interpretability (unavailable to the MLP), "
            "this makes KAN-Route the preferable router whenever class imbalance or auditability "
            "matter — which is the typical production setting for an open, growing tool taxonomy.\n\n")
    f.write(ablation_df_full.to_markdown(index=False))
print("✅ kan_vs_mlp_reframing.md updated with the stronger, evidence-backed framing")

In [ ]:
# ── Cell 25: Corrected per-class analysis, honestly framed ───────────────────
# Flag which classes are even measurable before drawing conclusions
merged["reliable"] = merged["support"] >= 10   # arbitrary but reasonable floor
reliable = merged[merged["reliable"]]
unreliable = merged[~merged["reliable"]]

print(f"Classes with support < 10 (unreliable F1 estimate): {len(unreliable)} / {len(merged)}")
print(unreliable[["support","f1-score_kan","f1-score_mlp","gap"]].to_string())

print(f"\nAmong RELIABLE classes (support >= 10):")
print(f"  Avg KAN advantage, below-median support: "
      f"{reliable[reliable['support'] < reliable['support'].median()]['gap'].mean():+.3f}")
print(f"  Avg KAN advantage, above-median support: "
      f"{reliable[reliable['support'] >= reliable['support'].median()]['gap'].mean():+.3f}")

merged.to_csv(OUT / "diagnostics" / "per_class_kan_vs_mlp_gap.csv")

In [ ]:
# ── Cell 26: Replace Cell 24's overclaimed writeup with the honest version ───
with open(OUT / "kan_vs_mlp" / "kan_vs_mlp_reframing.md", "w") as f:
    f.write("# KAN vs MLP — Reframed Trade-off (Qwen +aug, verified v2 checkpoints)\n\n")
    f.write("Top-1 accuracy is statistically tied between KAN-Route and the MLP baseline "
            "across all tested feature configurations. Macro F1 diverges substantially "
            "(0.966 vs 0.828 in the pixel-fused setting), but per-class inspection shows "
            "this is **not** a clean 'KAN handles rare classes better' pattern — the "
            "support-stratified gap is negligible (+0.017 vs -0.004). Instead, the two "
            "models fail on different specific classes: KAN clearly outperforms on "
            "`age_gender_estimation`, `background_removal`, and `logo_detection`, while "
            "MLP outperforms on `point_cloud_generation`. Several classes "
            "(`face_detection`, `text_detection_ocr`, `surface_normal_estimation`, "
            "`facial_expression_recognition`) have test support under 4 samples and are "
            "not meaningfully evaluable at all — a limitation of the paraphrase-augmented "
            "corpus's uneven per-class yield, not a model comparison result.\n\n")
    f.write("The defensible claim: KAN-Route matches MLP on Top-1 while achieving a "
            "meaningfully higher macro F1, driven by fewer catastrophic single-class "
            "failures — plus its intrinsic spline-based interpretability, unavailable "
            "to the MLP. The macro-F1 gap should not be attributed to class-imbalance "
            "robustness without further controlled testing (e.g. a class-balanced "
            "resampled test set).\n\n")
    f.write("## Full ablation table\n\n")
    f.write(ablation_df_full.to_markdown(index=False))
    f.write("\n\n## Per-class gap (rarest classes first)\n\n")
    f.write(merged.reset_index().to_markdown(index=False))

print("✅ kan_vs_mlp_reframing.md corrected and saved")
print("✅ per_class_kan_vs_mlp_gap.csv saved to diagnostics/")

In [ ]:
# ── Cell 27: Final, precise version of the writeup (reliable-only split) ─────
with open(OUT / "kan_vs_mlp" / "kan_vs_mlp_reframing.md", "w") as f:
    f.write("# KAN vs MLP — Reframed Trade-off (Qwen +aug, verified v2 checkpoints)\n\n")
    f.write("Top-1 accuracy is statistically tied between KAN-Route and the MLP baseline "
            "across all tested feature configurations (97.58% both, pixel-fused setting). "
            "Macro F1 diverges more substantially (0.966 vs 0.828).\n\n")
    f.write("**Per-class analysis, correctly stratified:** 6 of 35 classes have test support "
            "under 10 samples (as low as n=1) and are not meaningfully evaluable — both models "
            "score 0.0 F1 on 4 of these regardless of architecture, a corpus coverage limitation "
            "from the paraphrase-augmentation pipeline's uneven per-class yield, not a genuine "
            "model comparison. Restricting to the 29 classes with support >= 10:\n\n")
    f.write("- Below-median-support classes: KAN advantage averages **+0.046** macro F1\n")
    f.write("- Above-median-support classes: KAN advantage averages **-0.005** macro F1 (~tied)\n\n")
    f.write("**Defensible claim:** KAN-Route matches MLP on Top-1 accuracy while showing a "
            "modest but consistent macro-F1 advantage concentrated in lower-support classes, "
            "on top of its intrinsic spline-based interpretability (unavailable to the MLP). "
            "The effect size is small (+0.046) and should be described as a modest trend, not "
            "a strong claim — a class-balanced resampled test set would be needed to confirm "
            "it holds beyond this specific split.\n\n")
    f.write("## Full ablation table\n\n")
    f.write(ablation_df_full.to_markdown(index=False))
    f.write("\n\n## Per-class gap, reliable classes only (support >= 10)\n\n")
    f.write(reliable.reset_index().sort_values("support").to_markdown(index=False))
    f.write("\n\n## Excluded — insufficient support (n < 10)\n\n")
    f.write(unreliable.reset_index().to_markdown(index=False))

print("✅ kan_vs_mlp_reframing.md — final version saved")

In [ ]:
# ── Cell 28: Confirm everything is in the output tree before zipping ─────────
print("Current outputs/ contents:")
for sub in sorted(OUT.iterdir()):
    if sub.is_dir():
        files = list(sub.rglob("*"))
        print(f"  {sub.name}/  ({len([f for f in files if f.is_file()])} files)")
        for f in sorted(files):
            if f.is_file():
                print(f"    - {f.relative_to(OUT)}")

In [ ]:
# ── Cell 31: Pull forward reusable assets from the try2qwen Kaggle dataset ───
TRY2QWEN = Path("/kaggle/input/datasets/deboparna9das/try2qwen")  # adjust slug if different
if not TRY2QWEN.exists():
    candidates = list(Path("/kaggle/input").glob("try2qwen*"))
    TRY2QWEN = candidates[0] if candidates else None
print(f"try2qwen located at: {TRY2QWEN}")

# Find the fuller of the two output dumps (the 0306 one — matches what we already inspected)
prev_out_candidates = list(TRY2QWEN.rglob("pixel_clip_results.csv"))
PREV_OUT_DIR = prev_out_candidates[0].parent.parent if prev_out_candidates else None
print(f"Previous outputs root: {PREV_OUT_DIR}")

# 1. Reused CSVs — safe as-is
for fname, dest in [
    ("qwen_final_results.csv", OUT / "reused_from_previous_run" / "qwen_final_results.csv"),
    ("qwen_training_histories.json", OUT / "reused_from_previous_run" / "qwen_training_histories.json"),
]:
    src = next(TRY2QWEN.rglob(fname), None)
    if src:
        shutil.copy(src, dest)
        print(f"✅ {fname}")

# 2. Cached proxy-CLIP embeddings — skip recomputation
src = next(TRY2QWEN.rglob("proxy_clip_text_embeddings_aug.npy"), None)
if src:
    shutil.copy(src, OUT / "pixel_clip" / "proxy_clip_text_embeddings_aug.npy")
    print("✅ proxy_clip_text_embeddings_aug.npy (skips CLIP re-encode)")

# 3. The three valid ablation checkpoints + their rows
valid_variants = ["TextOnly-MiniLM", "ProxyVisual-CLIPText", "Text+ProxyVisual"]
for pt_file in TRY2QWEN.rglob("*_best.pt"):
    if any(v in pt_file.name for v in valid_variants):
        shutil.copy(pt_file, OUT / "pixel_clip" / pt_file.name)
        print(f"✅ {pt_file.name}")

old_ablation = pd.read_csv(PREV_OUT_DIR / "pixel_clip" / "pixel_clip_results.csv")
old_ablation_valid = old_ablation[old_ablation["variant"].isin(valid_variants)]
old_ablation_valid.to_csv(OUT / "pixel_clip" / "pixel_clip_results.csv", index=False)
print(f"\n✅ Carried forward {len(old_ablation_valid)} valid ablation rows (excluding stale Text+PixelVisual)")
print(old_ablation_valid.to_string(index=False))

# 4. Latency — safe as-is
src = next(TRY2QWEN.rglob("fair_latency_results.csv"), None)
if src:
    shutil.copy(src, OUT / "latency" / "fair_latency_results.csv")
    print("✅ fair_latency_results.csv")

print("\n⚠️  NOT carried forward (stale — built from the broken checkpoint):")
print("   confusion matrices, per-class reports, kan_vs_mlp table,")
print("   interpretability audit, taxonomy overlap — these still need")
print("   regenerating against kan_model_v2/mlp_model_v2 (Cell 30 from before).")

In [ ]:
# ── Cell 32: Re-append Text+PixelVisual row + regenerate the two stale files ─

# 1. Re-append the Text+PixelVisual row (kan_model/mlp_model are still v2 in memory)
kan_p2, kan_l2 = evaluate_full(kan_model, test_loader, "KAN-Route (v2, confirm)")
mlp_p2, mlp_l2 = evaluate_full(mlp_model, test_loader, "MLP (v2, confirm)")

pixel_visual_row = pd.DataFrame([
    dict(variant="Text+PixelVisual", model="KAN-Route", dim=896,
         params=kan_model.count_parameters(), top1=float(np.mean(kan_p2==kan_l2)),
         macro_f1=f1_score(kan_l2, kan_p2, average="macro", zero_division=0)),
    dict(variant="Text+PixelVisual", model="MLP", dim=896,
         params=mlp_model.count_parameters(), top1=float(np.mean(mlp_p2==mlp_l2)),
         macro_f1=f1_score(mlp_l2, mlp_p2, average="macro", zero_division=0)),
])
current_table = pd.read_csv(OUT / "pixel_clip" / "pixel_clip_results.csv")
full_table = pd.concat([current_table, pixel_visual_row], ignore_index=True)
full_table.to_csv(OUT / "pixel_clip" / "pixel_clip_results.csv", index=False)
print("✅ pixel_clip_results.csv — full 4-variant table restored:")
print(full_table.to_string(index=False))

# 2. Regenerate interpretability audit against v2
_, _, _, test_idx_v2 = make_split_loaders(X_aug, y_aug)
def get_case_v2(idx):
    x = torch.tensor(X_aug[idx:idx+1], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(kan_model(x), dim=-1).cpu().numpy()[0]
    top3_idx = probs.argsort()[::-1][:3]
    gt, pred = int(y_aug[idx]), int(top3_idx[0])
    return dict(row_idx=idx, query=aug_data[idx]["query"],
                gt_tool=aug_data[idx]["tool"], pred_tool=le.classes_[pred],
                correct=(gt==pred), confidence=float(probs[pred]),
                margin=float(probs[top3_idx[0]]-probs[top3_idx[1]]),
                top3=[le.classes_[i] for i in top3_idx])

cases_v2 = [get_case_v2(i) for i in test_idx_v2]
correct_v2 = [c for c in cases_v2 if c["correct"]]
wrong_v2   = [c for c in cases_v2 if not c["correct"]]
audit_v2 = (sorted(correct_v2, key=lambda c: -c["margin"])[:10]
            + sorted(correct_v2, key=lambda c: c["margin"])[:10]
            + sorted(wrong_v2, key=lambda c: -c["confidence"])[:10])
for c, tag in zip(audit_v2, ["correct_easy"]*10+["correct_ambiguous"]*10+["misclassified"]*10):
    c["case_type"] = tag; c["notes"] = ""
pd.DataFrame(audit_v2).to_csv(OUT / "interpretability" / "interpretability_audit_cases.csv", index=False)
print("\n✅ interpretability_audit_cases.csv regenerated against v2")

# 3. Regenerate taxonomy overlap against v2
cm_full_v2 = confusion_matrix(kan_l2, kan_p2, labels=list(range(NUM_TOOLS)))
overlap_pairs = [("background_removal","image_inpainting"),("image_captioning","visual_question_answering"),
                 ("object_detection","image_classification"),("semantic_segmentation","matting"),
                 ("scene_recognition","image_captioning"),("instance_segmentation","panoptic_segmentation")]
rows = []
for a, b in overlap_pairs:
    if a in le.classes_ and b in le.classes_:
        ia, ib = list(le.classes_).index(a), list(le.classes_).index(b)
        rows.append(dict(pair=f"{a} vs {b}", a_to_b=int(cm_full_v2[ia,ib]), b_to_a=int(cm_full_v2[ib,ia]),
                          support_a=int(cm_full_v2[ia].sum()), support_b=int(cm_full_v2[ib].sum())))
pd.DataFrame(rows).to_csv(OUT / "diagnostics" / "taxonomy_overlap_analysis.csv", index=False)
print("✅ taxonomy_overlap_analysis.csv regenerated against v2")

In [ ]:
# ── Cell 33: Fix — restrict predictions to the 35 valid trained classes ──────
NUM_VALID = len(le.classes_)  # 35

def get_case_v2(idx):
    x = torch.tensor(X_aug[idx:idx+1], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        probs_full = torch.softmax(kan_model(x), dim=-1).cpu().numpy()[0]
    probs = probs_full[:NUM_VALID]          # ← ignore the 15 untrained output slots
    top3_idx = probs.argsort()[::-1][:3]
    gt, pred = int(y_aug[idx]), int(top3_idx[0])
    return dict(row_idx=idx, query=aug_data[idx]["query"],
                gt_tool=aug_data[idx]["tool"], pred_tool=le.classes_[pred],
                correct=(gt == pred), confidence=float(probs[pred]),
                margin=float(probs[top3_idx[0]] - probs[top3_idx[1]]),
                top3=[le.classes_[i] for i in top3_idx])

cases_v2 = [get_case_v2(i) for i in test_idx_v2]
correct_v2 = [c for c in cases_v2 if c["correct"]]
wrong_v2   = [c for c in cases_v2 if not c["correct"]]
audit_v2 = (sorted(correct_v2, key=lambda c: -c["margin"])[:10]
            + sorted(correct_v2, key=lambda c: c["margin"])[:10]
            + sorted(wrong_v2, key=lambda c: -c["confidence"])[:10])
for c, tag in zip(audit_v2, ["correct_easy"]*10 + ["correct_ambiguous"]*10 + ["misclassified"]*10):
    c["case_type"] = tag; c["notes"] = ""
pd.DataFrame(audit_v2).to_csv(OUT / "interpretability" / "interpretability_audit_cases.csv", index=False)
print(f"✅ interpretability_audit_cases.csv regenerated ({len(wrong_v2)} misclassified out of {len(cases_v2)} test cases)")

In [ ]:
# ── Cell 34: Fix — decode using the ACTUAL tool_id scheme, no truncation ─────
# y_aug is sparse tool_id (0-41, not contiguous) — build the real id->name map
# straight from the data itself, matching exactly what the model trained on.
id_to_name = {item["tool_id"]: item["tool"] for item in aug_data}

def get_case_v2(idx):
    x = torch.tensor(X_aug[idx:idx+1], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(kan_model(x), dim=-1).cpu().numpy()[0]  # full 50-way, no truncation
    top3_idx = probs.argsort()[::-1][:3]
    gt, pred = int(y_aug[idx]), int(top3_idx[0])
    return dict(row_idx=idx, query=aug_data[idx]["query"],
                gt_tool=aug_data[idx]["tool"],
                pred_tool=id_to_name.get(pred, f"<unused_id_{pred}>"),
                correct=(gt == pred), confidence=float(probs[pred]),
                margin=float(probs[top3_idx[0]] - probs[top3_idx[1]]),
                top3=[id_to_name.get(i, f"<unused_id_{i}>") for i in top3_idx])

cases_v2 = [get_case_v2(i) for i in test_idx_v2]
correct_v2 = [c for c in cases_v2 if c["correct"]]
wrong_v2   = [c for c in cases_v2 if not c["correct"]]
print(f"Sanity check: {len(correct_v2)}/{len(cases_v2)} correct "
      f"({len(correct_v2)/len(cases_v2)*100:.2f}%) — should now match ~97.58%")

audit_v2 = (sorted(correct_v2, key=lambda c: -c["margin"])[:10]
            + sorted(correct_v2, key=lambda c: c["margin"])[:10]
            + sorted(wrong_v2, key=lambda c: -c["confidence"])[:10])
for c, tag in zip(audit_v2, ["correct_easy"]*10 + ["correct_ambiguous"]*10 + ["misclassified"]*10):
    c["case_type"] = tag; c["notes"] = ""
pd.DataFrame(audit_v2).to_csv(OUT / "interpretability" / "interpretability_audit_cases.csv", index=False)
print("✅ interpretability_audit_cases.csv regenerated correctly")

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

In [ ]:
# ── Cell 35: Fix taxonomy overlap — same id_to_name mapping, no le.classes_ ──
name_to_id = {v: k for k, v in id_to_name.items()}
cm_full_v2 = confusion_matrix(kan_l2, kan_p2, labels=list(range(NUM_TOOLS)))

overlap_pairs = [("background_removal","image_inpainting"),("image_captioning","visual_question_answering"),
                 ("object_detection","image_classification"),("semantic_segmentation","matting"),
                 ("scene_recognition","image_captioning"),("instance_segmentation","panoptic_segmentation")]
rows = []
for a, b in overlap_pairs:
    if a in name_to_id and b in name_to_id:
        ia, ib = name_to_id[a], name_to_id[b]
        rows.append(dict(pair=f"{a} vs {b}", a_to_b=int(cm_full_v2[ia,ib]), b_to_a=int(cm_full_v2[ib,ia]),
                          support_a=int(cm_full_v2[ia].sum()), support_b=int(cm_full_v2[ib].sum())))
pd.DataFrame(rows).to_csv(OUT / "diagnostics" / "taxonomy_overlap_analysis.csv", index=False)
print("✅ taxonomy_overlap_analysis.csv corrected")

In [ ]:
# ── Cell 36: Verify kan_per_class_report.csv used correct names, not TOOLS ───
per_class = pd.read_csv(OUT / "pixel_clip" / "kan_per_class_report.csv", index_col=0)
report_names = set(per_class.index) - {"accuracy", "macro avg", "weighted avg"}
correct_names = set(id_to_name.values())

mismatched = report_names - correct_names
print(f"Names in per-class report NOT in the correct tool_id->name mapping: {mismatched if mismatched else 'none — ✅ report is correctly labeled'}")

In [ ]:
# ── Cell 28 (final run): directory listing before zip ─────────────────────────
print("Current outputs/ contents:")
for sub in sorted(OUT.iterdir()):
    if sub.is_dir():
        files = [f for f in sub.rglob("*") if f.is_file()]
        print(f"  {sub.name}/  ({len(files)} files)")
        for f in sorted(files):
            print(f"    - {f.relative_to(OUT)}")

In [ ]:
import zipfile
from pathlib import Path

def zip_dir(src_dir, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in src_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(src_dir))

full_zip = Path(
    f"/kaggle/working/qwen_journal_revision_outputs_FINAL_{timestamp}.zip"
)

zip_dir(OUT, full_zip)

print("\nDownload from the Kaggle 'Output' tab:", full_zip.name)

In [ ]:
# ── Cell 37: Fix the stale kan_mlp_final_ablation_table.csv ──────────────────
kan_p3, kan_l3 = evaluate_full(kan_model, test_loader, "KAN-Route (final CSV fix)")
mlp_p3, mlp_l3 = evaluate_full(mlp_model, test_loader, "MLP (final CSV fix)")

reframe_row = pd.DataFrame([{
    "setting": "Qwen 35-tool +aug (pixel-CLIP, v2 verified)",
    "kan_top1": np.mean(kan_p3 == kan_l3), "mlp_top1": np.mean(mlp_p3 == mlp_l3),
    "kan_macro_f1": f1_score(kan_l3, kan_p3, average="macro", zero_division=0),
    "mlp_macro_f1": f1_score(mlp_l3, mlp_p3, average="macro", zero_division=0),
    "kan_params": kan_model.count_parameters(), "mlp_params": mlp_model.count_parameters(),
    "interpretability": "KAN=spline curves, MLP=none",
}])
reframe_row.to_csv(OUT / "kan_vs_mlp" / "kan_mlp_final_ablation_table.csv", index=False)
print(reframe_row.T)

In [ ]:
# ── Final zip (corrected) ─────────────────────────────────────────────────────
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

def zip_dir(src_dir, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in src_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(src_dir.parent))
    print(f"✅ {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB)")

full_zip = Path(f"/kaggle/working/qwen_journal_revision_outputs_FINAL_v2_{timestamp}.zip")
zip_dir(OUT, full_zip)
print("\nDownload from the Kaggle 'Output' tab:", full_zip.name)